# MLflow Models
## Key terms
- **model Registry**: Centralized model storage to manage version and lifecycle
- **MLflow artifacts**: Files including model, conda YAML that details software environment
- **MLflow serve**: Expose package model via real-time inference REST API

## MLflow package components
- conda.yaml
- MLmodel
- python_env.yaml
- python_model.pkl
- requirements.txt

### MLmodel
```yaml
artifacts_path: model
flavors:
    python_funtion:
        cloudpickle_verion: 2.2.0
        env: conda.yaml
        loader_module: mlflow.pyfunc.model
        python_model:   python_model.pkl
        python_version: 3.8.13
mlflow_version: 1.28.0
model_uuid: d4e3qfd347634165b7148135c5f901c6
run_id: bee7a45b579948389dc60b2dfcc20192
signature:
    inputs: '[{"name": "text", "type": "string"}]'
    outputs: '[{"name": "text", "type": "string"}]'
utc_time_created: '2022-10-02 19:25:05.049141'
```

### conda.yaml
```yaml
channels:
- conda-forge
dependencies:
- python=3.8.13
- pip<=22.2.2
- pip:
  - mlflow
  - cloudpickle==2.2.0
  - dill==0.3.5.1
  - ipython==8.5.0
  - sentencepiece==0.1.97
  - torch==1.12.1
  - transformers==4.22.2
name: mlflow-env
```

### python_env.yaml
```yaml
python: 3.8.13
build_dependecies:
- pip==22.2.2
- sepuptools==65.3.0
- wheel==0.37.1
dependencies
--r requirements.txt
```

## Using a Registry with an MLflow Model
```python
from mlflow import MLflowClient

client = MLflowClient()
client.create_registred_model("onnx-t5")
```
- need to produce model during training
- `log_model`
- `mlflow server --backend-storage-uri sqllite:///mlflow.db --default-artifact-root /temp/ --host 127.0.0.1:500`
    - setting up some db for storing st

## Referencing Artifacts with the API
```python
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:500")

from mlflow import MLflowClient
client = MLflowClient()

# create a new registered model
# this model must not existing already
client.create_registed_model("t5-onnx")

# Delete a registred model
client.delete_registred_model("t5-onnx")

client.update_model_version(
    name = "t5-small-summarizer",
    version = 1,
    description = "This is the T5 model in an ONNX version 1.6 using Opset 12"
)
```

## Serving and Saving Model
```python
from transformers import pipeline
import pandas as pd
import os
import mlflow
from mlflow import log_artifact
from mlflow.model import ModelSignature
import json
from mlflow.tracking import MLflowClient

class Summarizer(mlflow.pyfunc.PythonModel):

    def __init__(self):
        from transformers import T5Tokenizer, T5ForConditionalGeneration

        self.tokenizer = T5Tokenizer
        self.model = T5ForConditionalGeneration.from_pretrained("t5-small")

    def predict(self, context, model_input):
        input_ids =  self.tokenizer(f"translate English to German: {model_input}", return_tensor="pt").input_ids
        ouputs = self.model.generate(input_ids)
        return self.tokenizer.decode(output[0], skip_sepecial_tokens=True)

model_inpiut = json.dumps([
    {
        'name': 'text',
        'type': 'string'
    }
])
model_output = json.dumps([
    {
        'name': 'text',
        'type': 'string'
    }
])
signature = ModelSignature.from_dict({
    'inputs': model_input,
    'ouputs': model_ouput
})

mlflow.set_tracking_uri("http://127.0.0.1:5000")

# start tracking
with mlflow.start_run(run_name="t5-summarizer") as run:
    print(run.info.run_id)
    runner = rn.info.run_id
    print(
        "mlflow model serve -m runs:/" +
        run.info.run_id + "/model --no-conda"
    )
    mlflow.pyfunc.log_model(
        'model', 
        loader_module=None, 
        data_path=None, 
        code_path=None, 
        conda_env = None,
        python_model = Summarizer(),
        artifacts = None,
        registered_model_name = 't5-sall-summarizer",
        signature = signature,
        input_example = None,
        awiat_retistration_for = 0
    )

# curl -X POST -H "Content-Type:application/json; format=pandas-split" --data '{"columns": ["text"], "data": [["Today is a perfect day to practice automation skills"]]}' http://127.0.0.1:5000/invocations
